# DUS Query Tree — Step-by-Step Walkthrough

This notebook traces how DUS (Domain Unified Symbolic) builds and explores its query tree on a tiny hand-crafted sample.  
DUS runs DUC (Domain Unified Constant) per domain; for a single-domain sample they are equivalent, so we instrument `discover_duc` directly.

**What you will see per step:**
- The query string being tested
- Whether it matches the sample (✓ / ✗)
- Which traces it matched
- The children queued for exploration next
- A live ASCII picture of the tree after each matching query is added

In [11]:
import sys
sys.path.insert(0, '../src')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Build a small sample

We use a hand-crafted 1-dimensional sample with 3 short traces so the tree stays small enough to read.

In [12]:
from sample_multidim import MultidimSample

# MultidimSample infers event dimension by counting semicolons per event token.
# The generator always appends a trailing ';', so 1-D events look like 'a;'
# not 'a'.  We follow the same convention here.
traces = [
    'a; b; a; c;',
    'a; b; b; c;',
    'a; a; b; c;',
]

sample = MultidimSample()
sample.set_sample(traces)
sample.calc_sample_typeset(calculate_all=True)   # populate _sample_typeset

print("Sample traces:")
for i, t in enumerate(sample._sample):
    print(f"  [{i}] {t}")
print(f"Event dimension : {sample._sample_event_dimension}")
print(f"Alphabet        : {sorted(sample._sample_typeset)}")

Sample traces:
  [0] a; b; a; c;
  [1] a; b; b; c;
  [2] a; a; b; c;
Event dimension : 1
Alphabet        : ['a', 'b', 'c']


## 2. Instrumented DUC / DUS

We copy the core BFS loop from `discover_duc` and add `print` calls at each decision point so you can watch the tree being built.

In [13]:
from collections import deque
from math import ceil

from query_multidim import MultidimQuery
from hyper_linked_tree import HyperLinkedTree
from discovery_shared import _next_queries_multidim, ht_descriptive_queries
from dus import discover_dus  # top-level DUS entry point


def _tree_lines(tree, indent=0, vertex=None):
    """Render the HyperLinkedTree as indented text lines."""
    lines = []
    if vertex is None:
        vertex = tree._root_vertex  # HyperLinkedTree stores root as _root_vertex
        label = "(root)"
    else:
        label = repr(vertex.query_string)
    prefix = "  " * indent
    lines.append(f"{prefix}{label}")
    for child in sorted(vertex.child_vertices, key=lambda v: v.query_string):
        lines.extend(_tree_lines(tree, indent + 1, child))
    return lines


def discover_dus_verbose(sample, supp: float, max_query_length: int = -1):
    """Step-by-step instrumented version of DUS.

    DUS routes based on domain count:
      - 1 domain  → runs DUC directly (same BFS loop below)
      - N domains → runs per-domain DUC then merges (discovery_shared._domain_separated_discovery)

    For this walkthrough we instrument the core BFS that DUS uses per domain.
    """
    domain_cnt = sample._sample_event_dimension
    print(f"DUS: {domain_cnt} domain(s) detected.")
    if domain_cnt == 1:
        print("DUS: single domain → running DUC directly (identical to DUS for 1-D).\n")
    else:
        print(f"DUS: multi-domain → would run DUC per domain then merge (showing domain 0 here).\n")

    if max_query_length == -1:
        threshold = ceil(sample._sample_size * supp)
        trace_lengths = sorted(len(t.split()) for t in sample._sample)
        max_query_length = trace_lengths[sample._sample_size - threshold]

    gen_event = ';' * domain_cnt
    gen_event_list = list(gen_event)

    att_vsdb = sample.get_att_vertical_sequence_database()
    sample_size = sample._sample_size
    vsdb = {}
    patternset = {}
    all_patternset = {}

    for domain, dom_vsdb in att_vsdb.items():
        patternset[domain] = set()
        all_patternset[domain] = {tid: set() for tid in range(sample_size)}
        for key, value in dom_vsdb.items():
            new_key = ''.join(gen_event_list[:domain] + [key] + gen_event_list[domain:])
            vsdb[new_key] = value
            for item in value.keys():
                if len(value[item]) >= 2:
                    all_patternset[domain][item].add(key)
                    patternset[domain].add(key)

    sample_sized_support = ceil(sample._sample_size * supp)
    alphabet = sorted(
        {sym for sym, val in vsdb.items() if len(val) >= sample_sized_support}
    )

    print(f"Support threshold : {supp}  ({sample_sized_support}/{sample_size} traces)")
    print(f"Max query length  : {max_query_length}")
    print(f"Alphabet          : {alphabet}")
    print(f"Patternset (vars) : {dict(patternset)}")
    print()

    query = MultidimQuery()
    query.set_query_string(gen_event)

    matching_dict = {gen_event: query}
    non_matching_dict = {}
    parent_dict = {gen_event: query}
    dict_iter = {}
    querycount = 1

    children = _next_queries_multidim(query, alphabet, max_query_length, patternset)
    parent_dict.update({child._query_string: query for child in children})
    stack = deque(children)

    query_tree = HyperLinkedTree(
        ceil(supp * sample._sample_size),
        event_dimension=domain_cnt,
    )

    step = 0
    separator = "-" * 60

    while stack:
        query = stack.pop()
        qs = query._query_string
        query.set_query_matchtest('smarter')
        querycount += 1
        step += 1

        parent = parent_dict[qs]
        parent_qs = parent._query_string

        matched = query.match_sample(
            sample=sample,
            supp=supp,
            dict_iter=dict_iter,
            patternset=all_patternset,
            parent_dict=parent_dict,
        )

        matched_traces = query._query_matched_traces
        status = "✓ MATCH" if matched else "✗ PRUNE"

        print(separator)
        print(f"Step {step:>3} | {status} | query: {repr(qs)}")
        print(f"         | parent: {repr(parent_qs)}")
        print(f"         | matched traces: {matched_traces}")

        if matched:
            matching_dict[qs] = query

            tree_parent_qs = '' if parent_qs == gen_event else parent_qs
            parent_vertex = query_tree.find_vertex(tree_parent_qs)
            if not query_tree.find_vertex(qs):
                vertex = query_tree.insert_query_string(
                    parent_vertex, qs, query=query, search_for_parents=False
                )
                vertex.matched_traces = matched_traces

            next_children = _next_queries_multidim(query, alphabet, max_query_length, patternset)
            child_strings = [c._query_string for c in next_children]
            print(f"         | children queued: {child_strings}")
            if next_children:
                stack.extend(next_children)
                parent_dict.update({c._query_string: query for c in next_children})

            print("\n  Current query tree:")
            for line in _tree_lines(query_tree):
                print(f"  {line}")
        else:
            non_matching_dict[qs] = query
            print(f"         | children queued: (none — branch pruned)")

    print(separator)
    print(f"\nDone. Explored {querycount} queries total.")

    queryset, query_tree = ht_descriptive_queries(query_tree, set(matching_dict.keys()))
    final_queryset = queryset - {gen_event}

    return {
        'queryset': final_queryset,
        'query_tree': query_tree,
        'matching_dict': matching_dict,
        'non_matching_dict': non_matching_dict,
    }

## 3. Run the walkthrough

In [14]:
SUPP = 1.0   # require every query to match ALL traces

result = discover_dus_verbose(sample, supp=SUPP)

DUS: 1 domain(s) detected.
DUS: single domain → running DUC directly (identical to DUS for 1-D).

Support threshold : 1.0  (3/3 traces)
Max query length  : 4
Alphabet          : ['a;', 'b;', 'c;']
Patternset (vars) : {0: {'a', 'b'}}

------------------------------------------------------------
Step   1 | ✓ MATCH | query: 'c;'
         | parent: ';'
         | matched traces: [0, 1, 2]
         | children queued: ['c; $x0; $x0;', 'c; a;', 'c; b;', 'c; c;']

  Current query tree:
  (root)
    'c;'
------------------------------------------------------------
Step   2 | ✗ PRUNE | query: 'c; c;'
         | parent: 'c;'
         | matched traces: []
         | children queued: (none — branch pruned)
------------------------------------------------------------
Step   3 | ✗ PRUNE | query: 'c; b;'
         | parent: 'c;'
         | matched traces: []
         | children queued: (none — branch pruned)
------------------------------------------------------------
Step   4 | ✗ PRUNE | query: 'c; a;

## 4. Final result summary

In [15]:
print(f"Matching queries   : {len(result['matching_dict'])}")
print(f"Non-matching       : {len(result['non_matching_dict'])}")
print()
print("Descriptive queryset (final output of DUS):")
for q in sorted(result['queryset']):
    print(f"  {q}")

Matching queries   : 10
Non-matching       : 35

Descriptive queryset (final output of DUS):
  $x0; $x0; c;
  a; b; c;


## 5. Try a lower support threshold

With `supp=0.67` (≥2/3 traces), more queries survive pruning and the tree grows larger.

In [ ]:
# Re-create the sample so match state is fresh
sample2 = MultidimSample()
sample2.set_sample(traces)
sample2.calc_sample_typeset(calculate_all=True)

result2 = discover_dus_verbose(sample2, supp=2/3)

In [ ]:
print("Descriptive queryset at supp=0.67:")
for q in sorted(result2['queryset']):
    print(f"  {q}")